In [ ]:
# # temp
# %pip install -r requirements.txt

In [ ]:
from pyinaturalist import *
import pandas as pd




ambystoma = 26721 # taxon id (Ambystoma genus)
states = { # place ids
    "OH": 31,
    "MI": 29,
    "IN": 20,
    "PA": 42,
    "KY": 26,
    "WV": 33
}


observations = {"results": [], "total_results": 0}
for state in states:
    response = get_observations(
        taxon_id=ambystoma,
        place_id=states[state],
        quality_grade='research',
        captive=False,   # exclude captive/zoo observations
        page="all",      # fetch all pages
    )
    for obs in response['results']:
        obs['state'] = state  # tag each record with its state
    observations['results'].extend(response['results'])
    observations['total_results'] += response['total_results']
    print(f"{state}: {response['total_results']} research-grade Ambystoma observations")

print(f"Total: {observations['total_results']} research-grade Ambystoma observations across {len(states)} states")
pprint(observations['results'][:5])


In [ ]:
df = pd.DataFrame(observations['results'])
df.head()

In [ ]:
df['species'] = df['taxon'].apply(lambda t: t['name'])

In [ ]:
df.head()

In [ ]:
# ── Parse latitude and longitude from the location field ──
# pyinaturalist converts the raw API 'location' string ('lat,lon') into a
# list of two floats [lat, lon] before returning it. We extract from that.
# The separate latitude/longitude fields may be None for geoprivacy=obscured
# observations, but location always has the (possibly obscured) coordinates.

def extract_coords(loc):
    """Extract (lat, lon) from pyinaturalist location field (list of floats or None)"""
    if loc is not None and isinstance(loc, (list, tuple)) and len(loc) == 2:
        return float(loc[0]), float(loc[1])
    return None, None

df['latitude'], df['longitude'] = zip(*df['location'].apply(extract_coords))

# Drop records with no coordinates
before = len(df)
df = df.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
after = len(df)
print(f'Records with coordinates: {after} / {before} (dropped {before - after})')
print(f'Lat range: {df.latitude.min():.4f} to {df.latitude.max():.4f}')
print(f'Lon range: {df.longitude.min():.4f} to {df.longitude.max():.4f}')

In [ ]:
# ── NLCD Land Cover (30m resolution) ──
# Uses pygeohydro (part of HyRiver ecosystem) to query NLCD via MRLC GeoServer
# Returns NLCD land cover class codes at each observation point
#
# NLCD class codes we care about:
#   11  Open Water
#   21  Developed, Open Space
#   22  Developed, Low Intensity
#   23  Developed, Medium Intensity
#   24  Developed, High Intensity
#   31  Barren Land
#   41  Deciduous Forest
#   42  Evergreen Forest
#   43  Mixed Forest
#   52  Shrub/Scrub
#   71  Grassland/Herbaceous
#   81  Pasture/Hay
#   82  Cultivated Crops
#   90  Woody Wetlands
#   95  Emergent Herbaceous Wetlands

import pygeohydro as gh

# nlcd_bycoords takes a list of (lon, lat) tuples
# Process in batches of 500 to avoid server limits
BATCH_SIZE = 500
coords = list(zip(df['longitude'], df['latitude']))

nlcd_values = []
for i in range(0, len(coords), BATCH_SIZE):
    batch = coords[i:i + BATCH_SIZE]
    print(f'  NLCD batch {i//BATCH_SIZE + 1}/{(len(coords)-1)//BATCH_SIZE + 1} ({len(batch)} points)...')
    result = gh.nlcd_bycoords(batch, years={'cover': [2021]})
    nlcd_values.extend(result['cover_2021'].tolist())

df['nlcd_landcover'] = nlcd_values
print(f'\nNLCD land cover extracted for {len(nlcd_values)} points')
print(df['nlcd_landcover'].value_counts().sort_index())

In [ ]:
# Map NLCD codes to human-readable names
NLCD_NAMES = {
    11: 'Open Water',
    21: 'Developed, Open Space',
    22: 'Developed, Low Intensity',
    23: 'Developed, Medium Intensity',
    24: 'Developed, High Intensity',
    31: 'Barren Land',
    41: 'Deciduous Forest',
    42: 'Evergreen Forest',
    43: 'Mixed Forest',
    52: 'Shrub/Scrub',
    71: 'Grassland/Herbaceous',
    81: 'Pasture/Hay',
    82: 'Cultivated Crops',
    90: 'Woody Wetlands',
    95: 'Emergent Herbaceous Wetlands',
}

df['nlcd_landcover_name'] = df['nlcd_landcover'].map(NLCD_NAMES)
print(df['nlcd_landcover_name'].value_counts())

In [ ]:
# ── SSURGO Soil Data via SoilPoint API ──
# Returns: soil series, texture, drainage class, hydrologic group,
#          available water capacity (AWC), pH, organic matter
# API: https://soilpoint.dev/v1/soil?lat=X&lon=Y
# Free, cached, hits USDA-NRCS SSURGO via Soil Data Access

import requests
import time

soil_data = []
failed = 0
for idx, row in df.iterrows():
    if idx % 500 == 0:
        print(f'  Soil query {idx}/{len(df)}...')
    try:
        resp = requests.get(
            'https://soilpoint.dev/v1/soil',
            params={'lat': row['latitude'], 'lon': row['longitude']},
            timeout=15,
        )
        resp.raise_for_status()
        soil = resp.json()
        soil_data.append({
            'soil_series': soil.get('soilSeries'),
            'soil_texture': soil.get('texture'),
            'soil_drainage': soil.get('drainageClass'),
            'soil_hydrologic_group': soil.get('hydrologicGroup'),
            'soil_awc': soil.get('awc'),
            'soil_ph': soil.get('ph'),
            'soil_organic_matter': soil.get('organicMatter'),
        })
    except Exception as e:
        failed += 1
        soil_data.append({
            'soil_series': None, 'soil_texture': None, 'soil_drainage': None,
            'soil_hydrologic_group': None, 'soil_awc': None,
            'soil_ph': None, 'soil_organic_matter': None,
        })
    # Be nice to the API — 10 req/sec is plenty
    time.sleep(0.1)

soil_df = pd.DataFrame(soil_data)
print(f'\nSoil data extracted: {len(soil_df)} records ({failed} failed)')
print(soil_df.head())

In [ ]:
# ── WorldClim 2.1 Bioclim Variables ──
# Downloads the bioclim ZIP archive, extracts the 5 selected variables,
# then samples values at each observation point using rasterio.
#
# NOTE: WorldClim reorganized their server. Individual .tif files are no
# longer available — only ZIP archives. The 30s (~1km) ZIP is 9.7 GB.
# We use the 5m (~4.5km) resolution instead (171 MB ZIP). This is adequate
# for climate covariates, which vary smoothly over space — the difference
# between 1km and 4.5km for annual mean temperature is negligible.
#
# Selected variables (minimize collinearity, maximize ecological relevance):
#   BIO1  - Annual Mean Temperature
#   BIO4  - Temperature Seasonality (standard deviation * 100)
#   BIO12 - Annual Precipitation
#   BIO15 - Precipitation Seasonality (coefficient of variation)
#   BIO18 - Precipitation of Warmest Quarter

import rasterio
import os
import urllib.request
import zipfile

BIO_VARS = [1, 4, 12, 15, 18]
WORLDCLIM_DIR = 'data/worldclim'
ZIP_PATH = os.path.join(WORLDCLIM_DIR, 'wc2.1_5m_bio.zip')
os.makedirs(WORLDCLIM_DIR, exist_ok=True)

# Download the bioclim ZIP if not already present
# URL: https://geodata.ucdavis.edu/climate/worldclim/2_1/base/wc2.1_5m_bio.zip
# Size: ~171 MB. Contains all 19 bioclim variables at 5 arc-min resolution.
if not os.path.exists(ZIP_PATH) or os.path.getsize(ZIP_PATH) < 1000000:
    url = 'https://geodata.ucdavis.edu/climate/worldclim/2_1/base/wc2.1_5m_bio.zip'
    print(f'Downloading WorldClim 5m bioclim ZIP ({url})...')
    print(f'File size: ~171 MB. This may take a minute...')
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=300) as resp:
                with open(ZIP_PATH, 'wb') as f:
                    f.write(resp.read())
            size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
            print(f'  Downloaded ({size_mb:.1f} MB)')
            break
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
            if os.path.exists(ZIP_PATH):
                os.remove(ZIP_PATH)
            if attempt == 2:
                raise
else:
    print(f'WorldClim ZIP already downloaded ({os.path.getsize(ZIP_PATH) / (1024*1024):.1f} MB)')

# Extract only the bioclim variables we need from the ZIP
# Files inside ZIP are named: wc2.1_5m_bio_1.tif, wc2.1_5m_bio_4.tif, etc.
print('Extracting bioclim rasters...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    for var in BIO_VARS:
        tif_name = f'wc2.1_5m_bio_{var}.tif'
        tif_path = os.path.join(WORLDCLIM_DIR, tif_name)
        if os.path.exists(tif_path):
            continue
        # Find the file in the ZIP (may be in a subdirectory)
        matching = [n for n in zf.namelist() if n.endswith(tif_name)]
        if matching:
            zf.extract(matching[0], WORLDCLIM_DIR)
            print(f'  Extracted BIO{var}')
        else:
            print(f'  WARNING: {tif_name} not found in ZIP')

# Extract values at each point
# WorldClim rasters are in EPSG:4326 (same as our lat/lon)
coords = list(zip(df['longitude'], df['latitude']))

for var in BIO_VARS:
    tif_path = os.path.join(WORLDCLIM_DIR, f'wc2.1_5m_bio_{var}.tif')
    col_name = f'bio{var}'
    print(f'  Extracting BIO{var}...')
    with rasterio.open(tif_path) as src:
        values = list(src.sample(coords))
        df[col_name] = [v[0] for v in values]

print(f'\nWorldClim extraction complete. Columns: {[f"bio{v}" for v in BIO_VARS]}')
print(df[[f'bio{v}' for v in BIO_VARS]].describe())

In [ ]:
# ── Elevation / Topography (USGS 3DEP / NED) ──
# Uses py3dep (part of HyRiver) to fetch elevation at point coordinates.
# If py3dep is not available, falls back to reading a downloaded DEM raster.
#
# Variables extracted:
#   elevation (meters)
# We can derive slope and TWI later from a DEM raster if needed.

try:
    import py3dep
    print('Using py3dep for elevation data...')
    coords = list(zip(df['longitude'], df['latitude']))
    
    # py3dep.elevation_bycoords returns elevation in meters
    # Process in batches to be gentle on the API
    BATCH = 500
    elevations = []
    for i in range(0, len(coords), BATCH):
        batch = coords[i:i + BATCH]
        print(f'  Elevation batch {i//BATCH + 1}...')
        elevs = py3dep.elevation_bycoords(batch, crs=4326)
        elevations.extend(elevs)
    df['elevation_m'] = elevations
    print(f'Elevation extracted for {len(elevations)} points')
    print(df['elevation_m'].describe())
except ImportError:
    print('py3dep not installed. Install with: pip install py3dep')
    print('Alternatively, download a DEM GeoTIFF from the USGS National Map')
    print('and extract values with rasterio, same as the WorldClim cells above.')
    df['elevation_m'] = None

In [ ]:
# ── Distance to Nearest Stream (NHD Flowline) ──
# Fetches NHD flowlines for the 6-state region via pynhd.WaterData,
# then computes distance from each observation point to the nearest stream.
#
# The 6-state area is split into 6 sub-regions because a single query
# for the full bounding box times out (too many flowlines). Each sub-region
# takes ~30-90 seconds. Total download: ~460K flowlines, ~5 minutes.
#
# Distance to streams matters for Ambystoma:
#   - Most species breed in vernal pools near streams/wetlands
#   - A. barbouri uses headwater streams directly

import pynhd
import geopandas as gpd
from shapely.geometry import Point, box
import time

# Split the 6-state bounding box into 6 sub-regions
# Full extent: (min_lon, min_lat) = (-90.4, 36.5) to (max_lon, max_lat) = (-74.7, 48.3)
min_lon, min_lat = -90.4, 36.5
max_lon, max_lat = -74.7, 48.3
lon_step = (max_lon - min_lon) / 3
lat_step = (max_lat - min_lat) / 2

sub_bboxes = []
for i_split in range(3):
    for j_split in range(2):
        x0 = min_lon + i_split * lon_step
        y0 = min_lat + j_split * lat_step
        sub_bboxes.append((x0, y0, x0 + lon_step, y0 + lat_step))

print(f'Fetching NHD flowlines in 6 sub-regions (~5 min total)...')
wd = pynhd.WaterData('nhdflowline_network')
all_flowlines = []
for i_box, sbox in enumerate(sub_bboxes):
    print(f'  Region {i_box+1}/6: ({sbox[0]:.1f}, {sbox[1]:.1f}) to ({sbox[2]:.1f}, {sbox[3]:.1f})...')
    try:
        fl = wd.bybox(sbox)
        print(f'    {len(fl)} flowlines')
        all_flowlines.append(fl)
    except Exception as e:
        print(f'    FAILED: {e}')
    time.sleep(1)

# Combine and deduplicate (flowlines that span region boundaries)
flowlines = gpd.GeoDataFrame(gpd.pd.concat(all_flowlines, ignore_index=True))
flowlines = flowlines.drop_duplicates(subset='comid')
print(f'\nTotal unique flowlines: {len(flowlines)}')

# Create GeoDataFrame of observation points
obs_points = gpd.GeoDataFrame(
    df[['latitude', 'longitude']],
    geometry=[Point(lon, lat) for lon, lat in zip(df['longitude'], df['latitude'])],
    crs='EPSG:4326',
)

# Project both to planar CRS for accurate distance in meters
# EPSG:5070 = NAD83 / Conus Albers
obs_proj = obs_points.to_crs('EPSG:5070')
flowlines_proj = flowlines.to_crs('EPSG:5070')

# Compute nearest-stream distance using spatial index (R-tree)
print('Computing nearest-stream distances...')
nearest = gpd.sjoin_nearest(
    obs_proj, 
    flowlines_proj[['geometry', 'comid']], 
    distance_col='dist_to_stream_m',
)

# Merge distance back into df (sjoin_nearest preserves index order)
df['dist_to_stream_m'] = nearest['dist_to_stream_m'].values
print(f'\nDistance to nearest stream computed for {len(nearest)} points')
print(f'Distance range: {df.dist_to_stream_m.min():.0f} to {df.dist_to_stream_m.max():.0f} m')
print(f'Median distance: {df.dist_to_stream_m.median():.0f} m')

In [ ]:
# ── Merge soil data into main df ──
for col in soil_df.columns:
    df[col] = soil_df[col].values

print(f'Merged soil data. DataFrame shape: {df.shape}')

In [ ]:
# ── Create the final covariates DataFrame ──
# Contains only: species, state, latitude, longitude, and covariate columns

covariate_cols = [
    # ── Identification ──
    'species', 'state', 'latitude', 'longitude',
    
    # ── NLCD Land Cover (30m) ──
    'nlcd_landcover',
    'nlcd_landcover_name',
    
    # ── SSURGO Soil ──
    'soil_series',
    'soil_texture',
    'soil_drainage',
    'soil_hydrologic_group',
    'soil_awc',
    'soil_ph',
    'soil_organic_matter',
    
    # ── WorldClim Bioclim (1km) ──
    'bio1',   # Annual Mean Temperature
    'bio4',   # Temperature Seasonality
    'bio12',  # Annual Precipitation
    'bio15',  # Precipitation Seasonality
    'bio18',  # Precipitation of Warmest Quarter
    
    # ── Topography ──
    'elevation_m',
    
    # ── Hydrology ──
    'dist_to_stream_m',
]

# Select only columns that exist in the DataFrame
existing_cols = [c for c in covariate_cols if c in df.columns]
missing_cols = [c for c in covariate_cols if c not in df.columns]
if missing_cols:
    print(f'Warning: missing columns (not yet extracted): {missing_cols}')

covariates_df = df[existing_cols].copy()
print(f'Final covariates DataFrame: {covariates_df.shape[0]} rows x {covariates_df.shape[1]} columns')
print(f'Columns: {list(covariates_df.columns)}')
covariates_df.head()

In [ ]:
# ── Save the covariates DataFrame to CSV ──
output_path = '/kaggle/working/ambystoma_covariates.pkl'
os.makedirs('data', exist_ok=True)
covariates_df.to_pickle(output_path, index=False)
print(f'Saved {len(covariates_df)} rows to {output_path}')
print(f'File size: {os.path.getsize(output_path) / (1024*1024):.1f} MB')

In [ ]:
# ── Quick summary of the covariate data ──
print('=== Covariate Summary ===')
print(f'Total observations: {len(covariates_df)}')
print(f'Species: {covariates_df.species.nunique()} taxa')
print(f'States: {covariates_df.state.nunique()} states')
print()
print('Species breakdown:')
print(covariates_df.species.value_counts())
print()
print('State breakdown:')
print(covariates_df.state.value_counts())
print()
print('Numeric covariates summary:')
numeric_cols = covariates_df.select_dtypes(include='number').columns.tolist()
print(covariates_df[numeric_cols].describe())
print()
print('Missing values per column:')
print(covariates_df.isnull().sum())